<a href="https://colab.research.google.com/github/sohailpayami2023/digital-signal-processing-python/blob/main/notebooks/00_python_foundations/03_scipy_signal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SciPy Signal Processing for MATLAB Users

`scipy.signal` provides the Python equivalent of the MATLAB
Signal Processing Toolbox: filter design, frequency analysis,
spectral estimation, and signal generation.

**Topics covered**
1. Imports and setup
2. Signal generation (chirp, square, sawtooth)
3. FIR filter design — `firwin`
4. IIR filter design — `butter`, `cheby1`
5. Frequency response — `freqz`
6. Applying filters — `lfilter` and `filtfilt`
7. Spectral estimation — `welch` and `periodogram`
8. MATLAB → SciPy quick reference


# 1. Imports and Setup

`scipy.signal` is a sub-module — import only what you need.
Functions used here mirror the MATLAB Signal Processing Toolbox.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Signal generation
from scipy.signal import chirp, square, sawtooth

# Filter design
from scipy.signal import firwin, butter, cheby1, iirnotch

# Frequency response
from scipy.signal import freqz

# Filter application
from scipy.signal import lfilter, filtfilt

# Spectral estimation
from scipy.signal import welch, periodogram, spectrogram

plt.rcParams.update({
    'figure.figsize' : (8, 3),
    'figure.dpi'     : 120,
    'axes.grid'      : True,
    'grid.alpha'     : 0.4,
    'lines.linewidth': 1.2,
})

# Sample rate used throughout this notebook
fs = 1000   # Hz


# 2. Signal Generation

SciPy provides ready-made waveforms beyond simple sinusoids.

| Waveform | MATLAB | SciPy |
|----------|--------|-------|
| Chirp | `chirp(t,f0,t1,f1)` | `chirp(t,f0,t1,f1)` |
| Square wave | `square(t)` | `square(t)` |
| Sawtooth | `sawtooth(t)` | `sawtooth(t)` |

> **Note:** Both MATLAB and SciPy `chirp()` use the same
> argument names. SciPy `square()` takes the angular frequency
> in radians, just like MATLAB.


In [ ]:
# Time axis: 1 second at fs = 1000 Hz
t = np.linspace(0, 1, fs, endpoint=False)

# Chirp: frequency sweeps from f0=50 Hz to f1=400 Hz
# at time t1=1 s.  MATLAB: chirp(t, 50, 1, 400)
ch = chirp(t, f0=50, t1=1, f1=400, method='linear')

# Square wave at 5 Hz — MATLAB: square(2*pi*5*t)
sq = square(2 * np.pi * 5 * t)

# Sawtooth at 5 Hz — MATLAB: sawtooth(2*pi*5*t)
sw = sawtooth(2 * np.pi * 5 * t)

fig, axes = plt.subplots(3, 1, figsize=(8, 5),
                         sharex=True)
axes[0].plot(t, ch)
axes[0].set(ylabel='Amplitude', title='Chirp (50→400 Hz)')
axes[1].plot(t, sq, color='C1')
axes[1].set(ylabel='Amplitude', title='Square wave (5 Hz)')
axes[2].plot(t, sw, color='C2')
axes[2].set(xlabel='Time (s)', ylabel='Amplitude',
            title='Sawtooth (5 Hz)')
plt.tight_layout()
plt.show()


# 3. FIR Filter Design — `firwin`

FIR (Finite Impulse Response) filters have a **linear phase**
response — every frequency component is delayed by the same
amount, which preserves waveform shape. This makes them
preferred for communications receivers.

`firwin(numtaps, cutoff, fs=fs)` designs a windowed-sinc
low-pass filter.

| Parameter | Meaning |
|-----------|----------|
| `numtaps` | Filter length (odd → symmetric, linear phase) |
| `cutoff` | Cut-off frequency in Hz (when `fs` is given) |
| `window` | Window type: `'hamming'` (default), `'hann'`, etc. |
| `pass_zero` | `True` → low-pass; `False` → high-pass |

MATLAB equivalent: `fir1(N, Wn, 'low')`
where `Wn = cutoff / (fs/2)` (normalised 0–1).


In [ ]:
# Design a 63-tap low-pass FIR filter, cut-off at 100 Hz
# MATLAB: b = fir1(62, 100/(fs/2));  (N = numtaps-1)
numtaps = 63
fc      = 100   # cut-off frequency in Hz

# firwin returns the filter coefficients (impulse response)
# window='hamming' reduces side-lobe levels to ~-41 dB
b_lp = firwin(numtaps, cutoff=fc, fs=fs, window='hamming')

print(f'Filter length : {len(b_lp)} taps')
print(f'Coefficients  : {b_lp[:5].round(5)} ...')

# Plot the impulse response — MATLAB: stem(b)
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.stem(b_lp)
ax.set(xlabel='Tap index', ylabel='Amplitude',
       title='FIR Impulse Response (63-tap, Hamming, LP 100 Hz)')
plt.tight_layout()
plt.show()


In [ ]:
# freqz: compute the frequency response of a filter
# MATLAB: [H, f] = freqz(b, 1, 1024, fs)
# worN = number of frequency points to evaluate
w, H = freqz(b_lp, worN=1024, fs=fs)

# w: frequency axis in Hz (when fs is given)
# H: complex frequency response
# 20*log10(|H|): magnitude in dB
H_dB = 20 * np.log10(np.abs(H) + 1e-12)

fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
axes[0].plot(w, H_dB)
axes[0].set(ylabel='Magnitude (dB)',
            title='FIR Frequency Response — LP 100 Hz')
axes[0].set_ylim(-80, 5)

# Phase response in degrees
phase_deg = np.angle(H, deg=True)
axes[1].plot(w, phase_deg, color='C1')
axes[1].set(xlabel='Frequency (Hz)', ylabel='Phase (deg)')
plt.tight_layout()
plt.show()


In [ ]:
# Band-pass FIR: pass 100–300 Hz, reject outside
# MATLAB: fir1(62, [100 300]/(fs/2))
# Pass a list of TWO cut-off frequencies for band-pass
b_bp = firwin(
    numtaps,
    cutoff=[100, 300],   # [low edge, high edge] in Hz
    pass_zero=False,     # False → band-pass (not low-pass)
    fs=fs,
)

w2, H2 = freqz(b_bp, worN=1024, fs=fs)
H2_dB = 20 * np.log10(np.abs(H2) + 1e-12)

fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(w2, H2_dB)
ax.set_ylim(-80, 5)
ax.set(xlabel='Frequency (Hz)', ylabel='Magnitude (dB)',
       title='FIR Band-pass 100–300 Hz')
plt.tight_layout()
plt.show()


# 4. IIR Filter Design — `butter` and `cheby1`

IIR (Infinite Impulse Response) filters achieve a given
roll-off with far fewer coefficients than FIR, but they
introduce **non-linear phase** — some frequencies are
delayed more than others, which can distort pulse shapes.

`butter(N, Wn, btype, fs)` designs a Butterworth filter:
- maximally flat passband (no ripple)
- roll-off of −20·N dB/decade past cut-off

`cheby1(N, rp, Wn, btype, fs)` designs a Chebyshev Type I:
- allows `rp` dB of ripple in the passband
- steeper roll-off than Butterworth for the same order

Both return `(b, a)` coefficient arrays.
MATLAB: `[b,a] = butter(N, Wn)` / `[b,a] = cheby1(N,rp,Wn)`


In [ ]:
# 5th-order Butterworth low-pass at 100 Hz
# MATLAB: [b,a] = butter(5, 100/(fs/2))
b_but, a_but = butter(5, Wn=100, btype='low', fs=fs)

# 5th-order Chebyshev Type I, 1 dB ripple, LP at 100 Hz
# MATLAB: [b,a] = cheby1(5, 1, 100/(fs/2))
b_ch1, a_ch1 = cheby1(5, rp=1, Wn=100,
                       btype='low', fs=fs)

# Frequency responses of both filters
w_b, H_b = freqz(b_but, a_but, worN=1024, fs=fs)
w_c, H_c = freqz(b_ch1, a_ch1, worN=1024, fs=fs)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(w_b, 20*np.log10(np.abs(H_b)+1e-12),
        label='Butterworth N=5')
ax.plot(w_c, 20*np.log10(np.abs(H_c)+1e-12),
        label='Chebyshev I  N=5, 1dB')
ax.set_ylim(-80, 5)
ax.set(xlabel='Frequency (Hz)', ylabel='Magnitude (dB)',
       title='IIR Filter Comparison — LP 100 Hz')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Notch filter: remove a single frequency (e.g. 50 Hz mains)
# MATLAB: [b,a] = iirnotch(f0/(fs/2), f0/(fs/2)/Q)
# iirnotch(w0, bw)  — w0 and bw as normalised fractions
f_notch = 50    # frequency to remove (Hz)
Q       = 30    # quality factor (higher = narrower notch)
b_n, a_n = iirnotch(w0=f_notch, bw=f_notch/Q, fs=fs)

w_n, H_n = freqz(b_n, a_n, worN=2048, fs=fs)

fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(w_n, 20*np.log10(np.abs(H_n)+1e-12))
ax.set_xlim(0, 200)
ax.set_ylim(-40, 5)
ax.set(xlabel='Frequency (Hz)', ylabel='Magnitude (dB)',
       title='Notch Filter — 50 Hz (Q=30)')
plt.tight_layout()
plt.show()


# 5. Frequency Response — `freqz`

`freqz(b, a, worN, fs)` evaluates the z-transform of the
filter `H(z) = B(z) / A(z)` on the unit circle at `worN`
equally-spaced frequency points.

- `b` — numerator coefficients (FIR: just `b`, `a=1`)
- `a` — denominator coefficients (FIR filters: `a = [1]`)
- `worN` — number of frequency points (default 512)
- `fs` — sample rate; when given, `w` is returned in Hz

MATLAB equivalent: `freqz(b, a, 1024, fs)`

Group delay = negative derivative of phase with respect to
frequency. For FIR filters it is constant (linear phase);
for IIR filters it varies across frequency.


In [ ]:
# Compare FIR (linear phase) vs IIR (non-linear phase)
# using the filters designed in sections 3 and 4.

w_fir, H_fir = freqz(b_lp, worN=1024, fs=fs)
w_iir, H_iir = freqz(b_but, a_but, worN=1024, fs=fs)

fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True)

# Magnitude
axes[0].plot(w_fir, 20*np.log10(np.abs(H_fir)+1e-12),
             label='FIR (63 taps)')
axes[0].plot(w_iir, 20*np.log10(np.abs(H_iir)+1e-12),
             label='Butterworth N=5', linestyle='--')
axes[0].set_ylim(-80, 5)
axes[0].set(ylabel='Magnitude (dB)',
            title='FIR vs IIR — Magnitude Response')
axes[0].legend()

# Phase
axes[1].plot(w_fir, np.unwrap(np.angle(H_fir)))
axes[1].plot(w_iir, np.unwrap(np.angle(H_iir)),
             linestyle='--')
axes[1].set(xlabel='Frequency (Hz)',
            ylabel='Phase (rad)',
            title='Phase Response — FIR is linear, IIR is not')
plt.tight_layout()
plt.show()


# 6. Applying Filters — `lfilter` and `filtfilt`

| Function | Phase distortion | Causality | Use case |
|----------|-----------------|-----------|----------|
| `lfilter` | Yes (group delay) | Causal | Real-time |
| `filtfilt` | None | Non-causal | Offline |

`lfilter(b, a, x)` applies the filter in one direction
— the output is delayed by the group delay.
MATLAB: `filter(b, a, x)`

`filtfilt(b, a, x)` filters forward then backward,
cancelling the phase shift — useful for offline analysis.
MATLAB: `filtfilt(b, a, x)`


In [ ]:
# Build a noisy signal: 50 Hz tone + wideband noise
np.random.seed(0)
t = np.linspace(0, 1, fs, endpoint=False)

# Clean 50 Hz sine wave
x_clean = np.sin(2 * np.pi * 50 * t)

# Add Gaussian noise
x_noisy = x_clean + 1.5 * np.random.randn(len(t))

# Apply FIR low-pass (removes noise above 100 Hz)
# lfilter: causal — has group delay of (numtaps-1)/2 samples
y_lf = lfilter(b_lp, 1.0, x_noisy)

# filtfilt: zero-phase — no delay, better for offline use
y_ff = filtfilt(b_lp, 1.0, x_noisy)

fig, axes = plt.subplots(3, 1, figsize=(8, 5),
                         sharex=True)
axes[0].plot(t, x_noisy, alpha=0.7)
axes[0].set(ylabel='Amplitude', title='Noisy signal')

axes[1].plot(t, y_lf)
axes[1].plot(t, x_clean, 'k--', lw=0.8, label='clean')
axes[1].set(ylabel='Amplitude', title='lfilter (causal)')
axes[1].legend()

axes[2].plot(t, y_ff)
axes[2].plot(t, x_clean, 'k--', lw=0.8, label='clean')
axes[2].set(xlabel='Time (s)', ylabel='Amplitude',
            title='filtfilt (zero-phase)')
axes[2].legend()
plt.tight_layout()
plt.show()


# 7. Spectral Estimation — `welch` and `periodogram`

A plain FFT of a noisy signal gives a noisy spectrum.
Power Spectral Density (PSD) estimation averages multiple
FFT segments to reduce variance.

| Method | MATLAB | SciPy |
|--------|--------|-------|
| Welch PSD | `pwelch(x,[],[],[],fs)` | `welch(x, fs=fs)` |
| Periodogram | `periodogram(x,[],[],fs)` | `periodogram(x,fs=fs)` |
| Spectrogram | `spectrogram(x,win,hop,nfft,fs)` | `spectrogram(x,...)` |

Welch's method:
1. Split signal into overlapping segments
2. Window each segment (reduces spectral leakage)
3. Compute FFT of each segment
4. Average the squared magnitudes → PSD


In [ ]:
# Build a test signal: two tones at 100 Hz and 250 Hz
np.random.seed(1)
t = np.linspace(0, 4, 4*fs, endpoint=False)

# Signal: 100 Hz (power 1) + 250 Hz (power 0.25) + noise
x = (np.sin(2 * np.pi * 100 * t)
   + 0.5 * np.sin(2 * np.pi * 250 * t)
   + 0.3 * np.random.randn(len(t)))

# Welch PSD — MATLAB: pwelch(x, [], [], [], fs)
# nperseg: samples per FFT segment
f_w, P_w = welch(x, fs=fs, nperseg=512)

# Periodogram (single FFT of whole signal)
f_p, P_p = periodogram(x, fs=fs)

fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(f_p, P_p, alpha=0.4, label='Periodogram')
ax.semilogy(f_w, P_w, lw=2, label='Welch (nperseg=512)')
ax.set_xlim(0, 400)
ax.set(xlabel='Frequency (Hz)',
       ylabel='PSD (V²/Hz)',
       title='Welch vs Periodogram — 100 Hz + 250 Hz tones')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Spectrogram of a chirp — shows frequency vs time
# MATLAB: spectrogram(x, hann(128), 64, 256, fs, 'yaxis')
t2 = np.linspace(0, 2, 2*fs, endpoint=False)
x2 = chirp(t2, f0=20, t1=2, f1=480, method='linear')

# spectrogram(x, window, noverlap, nfft, fs)
# nperseg : FFT window length in samples
# noverlap: samples shared between adjacent windows
# nfft    : FFT size (zero-padded if > nperseg)
f_sg, t_sg, Sxx = spectrogram(
    x2,
    fs=fs,
    nperseg=128,
    noverlap=112,    # 87.5% overlap
    nfft=256,
    window='hann',
)

# Sxx: shape (freq_bins, time_frames)
# Convert power to dB: 10*log10(Sxx)
Sxx_dB = 10 * np.log10(Sxx + 1e-12)

fig, ax = plt.subplots(figsize=(8, 3))
# pcolormesh: colour plot of 2-D array
im = ax.pcolormesh(
    t_sg, f_sg, Sxx_dB, shading='gouraud', cmap='viridis'
)
fig.colorbar(im, ax=ax, label='Power (dB)')
ax.set(xlabel='Time (s)', ylabel='Frequency (Hz)',
       title='Spectrogram — Linear Chirp 20→480 Hz')
plt.tight_layout()
plt.show()


# 8. MATLAB → SciPy Quick Reference

| Operation | MATLAB | SciPy |
|-----------|--------|-------|
| Chirp | `chirp(t,f0,t1,f1)` | `chirp(t,f0,t1,f1)` |
| Square wave | `square(t)` | `square(t)` |
| FIR design | `fir1(N-1, Wn)` | `firwin(N, fc, fs=fs)` |
| Butterworth | `butter(N, Wn)` | `butter(N, fc, fs=fs)` |
| Chebyshev I | `cheby1(N,rp,Wn)` | `cheby1(N,rp,fc,fs=fs)` |
| Notch filter | `iirnotch(w0,bw)` | `iirnotch(f0,bw,fs=fs)` |
| Freq response | `freqz(b,a,n,fs)` | `freqz(b,a,worN,fs)` |
| Apply filter | `filter(b,a,x)` | `lfilter(b,a,x)` |
| Zero-phase | `filtfilt(b,a,x)` | `filtfilt(b,a,x)` |
| Welch PSD | `pwelch(x,[],[],[],fs)` | `welch(x,fs=fs)` |
| Periodogram | `periodogram(x,[],[],fs)` | `periodogram(x,fs=fs)` |
| Spectrogram | `spectrogram(x,w,h,n,fs)` | `spectrogram(x,fs=fs,...)` |
| Unwrap phase | `unwrap(phi)` | `np.unwrap(phi)` |
| Group delay | `grpdelay(b,a,n,fs)` | `group_delay((b,a),w,fs)` |
| Pole-zero | `zplane(b,a)` | manual with `np.roots` |
| Convolution | `conv(a,b)` | `np.convolve(a,b)` |
| Correlation | `xcorr(x,y)` | `np.correlate(x,y,'full')` |

**Normalised frequency note:**
MATLAB `butter` expects `Wn` as a fraction of the Nyquist
frequency (0–1 where 1 = fs/2).
SciPy `butter` with `fs=fs` accepts Hz directly — no
manual normalisation needed.
